# Phase 4A Lab - Mock Search/Pricing Tools

Mục tiêu: chạy shopping tool contracts bằng fixtures local.

Expected output chính: `deal_search()` trả product candidates; `estimate_price()`
trả estimated value, discount, deal score.

Safety: giữ `ENABLE_REAL_SEARCH=false` và `ENABLE_REAL_MODEL_CALLS=false`.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy tool tests

Command này kiểm tra schemas, fixtures, fallback pricing và worker integration
liên quan tool path.


In [ ]:
run(["uv", "run", "pytest", "tests/test_tools.py", "-q", "--tb=short"], timeout=180)


## 2. Gọi mock deal_search trực tiếp

Expected: danh sách sản phẩm mock từ Amazon/BestBuy nếu query match fixture.


In [ ]:
from backend.tools.deal_search.schemas import DealSearchInput
from backend.tools.deal_search.tool import deal_search

search_output = deal_search(DealSearchInput(query_en="gaming laptop", source="All", max_results_per_source=2))
print(f"products={len(search_output.products)}")
print(f"warnings={search_output.warnings}")
for product in search_output.products[:3]:
    print(product.model_dump(exclude={"raw_source_payload"}))


## 3. Gọi mock price estimator trực tiếp

Expected: estimate dùng fixture nếu có, nếu không thì fallback 10%.


In [ ]:
from backend.tools.price_estimator.schemas import PriceEstimateInput
from backend.tools.price_estimator.tool import estimate_price

if search_output.products:
    estimate = estimate_price(PriceEstimateInput(product=search_output.products[0]))
    print(estimate.model_dump())
else:
    print("No product to estimate.")


## 4. Cách đọc kết quả

- `deal_score` nằm trong `hot/good/ok/overpriced`.
- `warnings=[]` là tốt, nhưng warning fallback vẫn hợp lệ nếu fixture thiếu.
- Không có URL/price nào được bịa ngoài fixture product data.
